### Anuncios de Política Monetaria
Para scrapear y crear la base de anuncios de decisiones de política monetaria (2000-hoy).

Notas sobre el corpus:
- Hasta enero de 2008 el instrumento era el "corto"; desde entonces, la tasa objetivo (`policy_instrument`).
- Antes de 2003 no había calendario fijo: 2000-2002 solo tiene los anuncios en que el corto cambió.
- Desde 2021 cada anuncio termina con una tabla de pronósticos de inflación; se elimina con `strip_forecast_table`.
  Residuo conocido: en `banxico_apm_2021_12_16` el PDF pone parte de la tabla arriba de la página 2 (~22 números quedan),
  y 2002-2004 conservan algunas etiquetas de ejes de gráficas. Ambos son menores.

Descarga ~20 MB de PDFs a `data/raw/` (no vienen incluidos) y escribe en `data/processed/banxico/`: `banxico_anuncios_{metadata.csv, pages, documents, chunks}`.

**Fecha de corte.** `FECHA_CORTE` fija el corpus al de los resultados (decisiones hasta
septiembre de 2026). El scraper lee la página de Banxico *hoy*, así que sin el corte
entrarían anuncios nuevos que no tienen calificación del LLM y fallarían los `assert`.
Para actualizar el corpus: subir la fecha, quitar o ajustar los `assert` de conteo y
calificar los chunks nuevos con `scripts/score_chunks_openrouter.py` (cuesta dinero).

In [1]:
import sys
from pathlib import Path

import pandas as pd

# El código vive en src/, agrupado en corpus/ index/ analysis/ (sin __init__.py):
# hay que agregar cada subcarpeta a la ruta de Python. Ver src/paths.py.
sys.path.insert(0, str(Path.cwd().parent / "src"))
import paths
paths.add_src_subfolders_to_path()

# Lo genérico (descargar páginas y PDFs) vive en corpus/scraping.py; lo específico de
# los anuncios (URL, cómo leer el título, formato del id, tabla de pronósticos), en su módulo.
from scraping import download_pdfs
from banxico_anuncios import scrape_index, strip_forecast_table
from processing import extract_pdf_pages, flag_page_quality
from chunking import create_document_chunks, split_chunks_by_tokens

In [2]:
PDF_DIR = paths.RAW_DIR / "anuncios_politica_monetaria" / "pdfs"
PROCESSED_DIR = paths.PROCESSED_DIR

PDF_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FECHA_CORTE = "2026-09-30"   # ver nota arriba

In [3]:
metadata = scrape_index()
print("en la página de Banxico hoy:", metadata.shape)

# solo las decisiones hasta la fecha de corte (los anuncios se fechan el día de la decisión)
metadata = metadata[metadata["publication_date"] <= pd.Timestamp(FECHA_CORTE)].reset_index(drop=True)
print("hasta la fecha de corte:", metadata.shape)
metadata.head(10)

en la página de Banxico hoy: (239, 14)
hasta la fecha de corte: (238, 14)


,document_id,institution,document_type,document_variant,title,publication_date,publication_month,reference_year,quarter,decision,policy_instrument,resource_type,pdf_url,source_page_url
0,banxico_apm_2026_09_24,Banco de México,Anuncio de política monetaria,announcement,El objetivo para la Tasa de Interés Interbanca...,2026-09-24,2026-09,2026,3,hold,tasa_objetivo,Texto completo,https://www.banxico.org.mx/publicaciones-y-pre...,https://www.banxico.org.mx/publicaciones-y-pre...
1,banxico_apm_2026_08_06,Banco de México,Anuncio de política monetaria,announcement,El objetivo para la Tasa de Interés Interbanca...,2026-08-06,2026-08,2026,3,hold,tasa_objetivo,Texto completo,https://www.banxico.org.mx/publicaciones-y-pre...,https://www.banxico.org.mx/publicaciones-y-pre...
2,banxico_apm_2026_06_25,Banco de México,Anuncio de política monetaria,announcement,El objetivo para la Tasa de Interés Interbanca...,2026-06-25,2026-06,2026,2,hold,tasa_objetivo,Texto completo,https://www.banxico.org.mx/publicaciones-y-pre...,https://www.banxico.org.mx/publicaciones-y-pre...
3,banxico_apm_2026_05_07,Banco de México,Anuncio de política monetaria,announcement,El objetivo para la Tasa de Interés Interbanca...,2026-05-07,2026-05,2026,2,ease,tasa_objetivo,Texto completo,https://www.banxico.org.mx/publicaciones-y-pre...,https://www.banxico.org.mx/publicaciones-y-pre...
4,banxico_apm_2026_03_26,Banco de México,Anuncio de política monetaria,announcement,El objetivo para la Tasa de Interés Interbanca...,2026-03-26,2026-03,2026,1,ease,tasa_objetivo,Texto completo,https://www.banxico.org.mx/publicaciones-y-pre...,https://www.banxico.org.mx/publicaciones-y-pre...
5,banxico_apm_2026_02_05,Banco de México,Anuncio de política monetaria,announcement,El objetivo para la Tasa de Interés Interbanca...,2026-02-05,2026-02,2026,1,hold,tasa_objetivo,Texto completo,https://www.banxico.org.mx/publicaciones-y-pre...,https://www.banxico.org.mx/publicaciones-y-pre...
6,banxico_apm_2025_12_18,Banco de México,Anuncio de política monetaria,announcement,El objetivo para la Tasa de Interés Interbanca...,2025-12-18,2025-12,2025,4,ease,tasa_objetivo,Texto completo,https://www.banxico.org.mx/publicaciones-y-pre...,https://www.banxico.org.mx/publicaciones-y-pre...
7,banxico_apm_2025_11_06,Banco de México,Anuncio de política monetaria,announcement,El objetivo para la Tasa de Interés Interbanca...,2025-11-06,2025-11,2025,4,ease,tasa_objetivo,Texto completo,https://www.banxico.org.mx/publicaciones-y-pre...,https://www.banxico.org.mx/publicaciones-y-pre...
8,banxico_apm_2025_09_25,Banco de México,Anuncio de política monetaria,announcement,El objetivo para la Tasa de Interés Interbanca...,2025-09-25,2025-09,2025,3,ease,tasa_objetivo,Texto completo,https://www.banxico.org.mx/publicaciones-y-pre...,https://www.banxico.org.mx/publicaciones-y-pre...
9,banxico_apm_2025_08_07,Banco de México,Anuncio de política monetaria,announcement,El objetivo para la Tasa de Interés Interbanca...,2025-08-07,2025-08,2025,3,ease,tasa_objetivo,Texto completo,https://www.banxico.org.mx/publicaciones-y-pre...,https://www.banxico.org.mx/publicaciones-y-pre...


In [4]:
# cuantos anuncios por año y con qué instrumento
pd.crosstab(metadata["reference_year"], metadata["policy_instrument"])

policy_instrument,corto,tasa_objetivo
reference_year,,
2000,6,0
2001,3,0
2002,4,0
2003,14,0
2004,14,0
2005,12,0
2006,12,0
2007,12,0
2008,1,10


In [5]:
metadata["decision"].value_counts(dropna=False)

decision
hold       129
tighten     60
ease        49
Name: count, dtype: int64

In [6]:
metadata_path = PROCESSED_DIR / "banxico_anuncios_metadata.csv"
metadata.to_csv(metadata_path, index=False)
print(metadata_path.name)

banxico_anuncios_metadata.csv


In [7]:
download_results = download_pdfs(metadata, PDF_DIR)
download_results["download_status"].value_counts()

  0%|          | 0/238 [00:00<?, ?it/s]

download_status
already_exists    238
Name: count, dtype: int64

In [8]:
metadata = metadata.merge(
    download_results,
    on="document_id",
    how="left",)

metadata.to_csv(metadata_path, index=False)

In [9]:
pages = extract_pdf_pages(metadata)
print(pages.shape)
pages.drop(columns="local_pdf_path").head()

  0%|          | 0/238 [00:00<?, ?it/s]

(459, 13)


,document_id,institution,document_type,title,publication_date,publication_month,reference_year,quarter,pdf_url,page_number,page_text,n_characters
0,banxico_apm_2026_09_24,Banco de México,Anuncio de política monetaria,El objetivo para la Tasa de Interés Interbanca...,2026-09-24,2026-09,2026,3,https://www.banxico.org.mx/publicaciones-y-pre...,1,24 de septiembre de 2026\n\nComunicado de Pren...,3529
1,banxico_apm_2026_09_24,Banco de México,Anuncio de política monetaria,El objetivo para la Tasa de Interés Interbanca...,2026-09-24,2026-09,2026,3,https://www.banxico.org.mx/publicaciones-y-pre...,2,INPC\n\nSubyacente\n\nMemorándum\n\nVariación ...,1686
2,banxico_apm_2026_08_06,Banco de México,Anuncio de política monetaria,El objetivo para la Tasa de Interés Interbanca...,2026-08-06,2026-08,2026,3,https://www.banxico.org.mx/publicaciones-y-pre...,1,6 de agosto de 2026\n\nComunicado de Prensa\n\...,3486
3,banxico_apm_2026_08_06,Banco de México,Anuncio de política monetaria,El objetivo para la Tasa de Interés Interbanca...,2026-08-06,2026-08,2026,3,https://www.banxico.org.mx/publicaciones-y-pre...,2,INPC\n\nSubyacente\n\nMemorándum\n\nVariación ...,1385
4,banxico_apm_2026_06_25,Banco de México,Anuncio de política monetaria,El objetivo para la Tasa de Interés Interbanca...,2026-06-25,2026-06,2026,2,https://www.banxico.org.mx/publicaciones-y-pre...,1,25 de junio de 2026\n\nComunicado de Prensa\n\...,3578


In [10]:
# quitar la tabla de pronósticos (2021 en adelante) y recalcular el largo
has_table = pages["page_text"].str.contains("ron[oó]sticos de la inflaci[oó]n general y subyacente", case=False, regex=True)
print("Pages with a forecast table:", has_table.sum())

pages["page_text"] = pages["page_text"].apply(strip_forecast_table)
pages["n_characters"] = pages["page_text"].str.len()

Pages with a forecast table: 42


In [11]:
assert metadata["document_id"].is_unique
assert metadata["pdf_url"].is_unique

assert metadata["document_id"].nunique() == 238
assert pages["document_id"].nunique() == 238

assert pages.duplicated(
    subset=["document_id", "page_number"]
).sum() == 0

In [12]:
pages = flag_page_quality(pages)
pages[
    [
        "is_near_empty",
        "is_short_page",
        "has_encoding_error",
        "is_usable_page",
    ]
].sum()

is_near_empty           3
is_short_page           3
has_encoding_error      0
is_usable_page        456
dtype: int64

In [13]:
pages_path = PROCESSED_DIR / "banxico_anuncios_pages.parquet"
pages.to_parquet(pages_path, index=False)
print(pages_path.name)

banxico_anuncios_pages.parquet


In [14]:
pages_for_chunks = (
    pages.loc[pages["is_usable_page"]]
    .sort_values(["document_id", "page_number"]).copy())

print("Pages in complete corpus:", len(pages))
print("Pages used for chunking:", len(pages_for_chunks))

assert pages_for_chunks["document_id"].nunique() == 238

Pages in complete corpus: 459
Pages used for chunking: 456


In [15]:
# base por documentos, una fila por anuncio
document_texts = (
    pages_for_chunks
    .groupby("document_id", as_index=False, sort=False)
    .agg(
        document_text=("page_text", lambda values: "\n\n".join(values)),
        n_pages=("page_number", "nunique"),
        total_characters=("n_characters", "sum"),
    )
)

document_metadata_columns = [
    "document_id",
    "institution",
    "document_type",
    "document_variant",
    "title",
    "publication_date",
    "publication_month",
    "reference_year",
    "quarter",
    "decision",
    "policy_instrument",
    "pdf_url",
]

documents = metadata[document_metadata_columns].merge(
    document_texts,
    on="document_id",
    how="left",
    validate="one_to_one",
)

documents["n_words"] = documents["document_text"].str.split().str.len()
documents.groupby("reference_year")["n_words"].median()

reference_year
2000     601.0
2001     702.0
2002     663.0
2003    1025.0
2004     503.0
2005     514.5
2006     556.5
2007     454.0
2008     417.0
2009     480.0
2010     571.0
2011     787.0
2012     974.5
2013    1074.0
2014    1033.0
2015    1215.5
2016    1403.0
2017    1708.5
2018    1358.5
2019    1528.0
2020     756.0
2021     715.5
2022     850.5
2023     770.0
2024     852.5
2025     760.0
2026     707.5
Name: n_words, dtype: float64

In [16]:
documents_path = PROCESSED_DIR / "banxico_anuncios_documents.parquet"
documents.to_parquet(documents_path, index=False)
print(documents_path.name)

banxico_anuncios_documents.parquet


In [17]:
# las columnas por defecto del chunker incluyen reference_period, que solo
# existe en los informes; aquí van las del anuncio
chunk_metadata_columns = [
    "institution",
    "document_type",
    "document_variant",
    "title",
    "publication_date",
    "publication_month",
    "reference_year",
    "quarter",
    "decision",
    "policy_instrument",
    "pdf_url",
]

chunks = create_document_chunks(
    pages_df=pages_for_chunks,
    metadata_df=metadata,
    metadata_columns=chunk_metadata_columns)

chunks.head()

  0%|          | 0/238 [00:00<?, ?it/s]

,chunk_id,document_id,chunk_number,page_start,page_end,chunk_text,n_characters,institution,document_type,document_variant,title,publication_date,publication_month,reference_year,quarter,decision,policy_instrument,pdf_url
0,banxico_apm_2000_01_18_c0001,banxico_apm_2000_01_18,1,1,2,BOLETÍN DE PRENSA\n\n• De diciembre de 1998 al...,3200,Banco de México,Anuncio de política monetaria,announcement,"El ""corto"" se aumenta a 180 millones de pesos",2000-01-18,2000-01,2000,1,tighten,corto,https://www.banxico.org.mx/publicaciones-y-pre...
1,banxico_apm_2000_05_16_c0001,banxico_apm_2000_05_16,1,1,2,descenso de la inflación anual continúe.\n\nco...,2969,Banco de México,Anuncio de política monetaria,announcement,"El ""corto"" se aumenta a 200 millones de pesos",2000-05-16,2000-05,2000,2,tighten,corto,https://www.banxico.org.mx/publicaciones-y-pre...
2,banxico_apm_2000_05_16_c0002,banxico_apm_2000_05_16,2,2,3,mundial genere un ajuste desordenado en los me...,901,Banco de México,Anuncio de política monetaria,announcement,"El ""corto"" se aumenta a 200 millones de pesos",2000-05-16,2000-05,2000,2,tighten,corto,https://www.banxico.org.mx/publicaciones-y-pre...
3,banxico_apm_2000_06_26_c0001,banxico_apm_2000_06_26,1,1,2,del 10%.\n\nfactores cabe destacar los siguien...,2616,Banco de México,Anuncio de política monetaria,announcement,"El ""corto"" se aumenta a 230 millones de pesos",2000-06-26,2000-06,2000,2,tighten,corto,https://www.banxico.org.mx/publicaciones-y-pre...
4,banxico_apm_2000_07_31_c0001,banxico_apm_2000_07_31,1,1,2,ciento.\n\nprincipales socios comerciales.\n\n...,2932,Banco de México,Anuncio de política monetaria,announcement,"El ""corto"" se aumenta a 280 millones de pesos",2000-07-31,2000-07,2000,3,tighten,corto,https://www.banxico.org.mx/publicaciones-y-pre...


In [18]:
## AGREGADO PARA PODER FIT EL MODELO DE EMBEDDING
from transformers import AutoTokenizer

MODEL_NAME = "intfloat/multilingual-e5-base"
MAX_TOKENS = 450

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

chunks = split_chunks_by_tokens(chunks, tokenizer, max_tokens=MAX_TOKENS)

print("New number of chunks:", len(chunks))

  0%|          | 0/578 [00:00<?, ?it/s]

New number of chunks: 960


In [19]:
chunks["n_tokens"] = chunks["chunk_text"].apply(
    lambda text: len(
        tokenizer(
            f"passage: {text}",
            add_special_tokens=True,
            truncation=False,
            verbose=False,
        )["input_ids"]
    )
)

print("Maximum tokens:", chunks["n_tokens"].max())
print("Chunks above 512 tokens:", (chunks["n_tokens"] > 512).sum())

Maximum tokens: 454
Chunks above 512 tokens: 0


In [20]:
# pocos chunks por anuncio: esto importa para la agregación de M2 (top decil)
chunks.groupby("document_id").size().describe()

count    238.000000
mean       4.033613
std        1.848401
min        1.000000
25%        3.000000
50%        3.500000
75%        5.000000
max        9.000000
dtype: float64

In [21]:
chunks_path = PROCESSED_DIR / "banxico_anuncios_chunks.parquet"
chunks.to_parquet(chunks_path, index=False)
print(chunks_path.name)

banxico_anuncios_chunks.parquet
